In [7]:
import pandas as pd
import numpy as np
import json
import seaborn as sns
import matplotlib.pyplot as plt

# Warning off
pd.options.mode.chained_assignment = None

import warnings
warnings.filterwarnings("ignore")

In [13]:
df = pd.read_csv('/home/sunaybhat/results_EBM_Defense/From_Scratch/Narcissus/Results.csv')

# Avg of last 8 End Acc
print(f'Nat Acc Avg: {df.iloc[-16:-8,:]["End Acc"].mean():.2f}% | Poison Acc Avg: {df.iloc[-16:-8,:]["P1 Acc"].mean():.2%}')

# print Max poison acc
print(f'Max Poison Acc: {df.iloc[-16:-8,:]["P1 Acc"].max():.2%}')

Nat Acc Avg: 91.88% | Poison Acc Avg: 15.62%
Max Poison Acc: 65.06%


In [3]:
def process_filtered_df(results_df,df_def,defense,narcissus,header,header_idx):
    if narcissus:
        avg_poison_success = df_def['P1 Acc'].mean() * 100
        std_poison_success = df_def['P1 Acc'].std() * 100
        avg_nat_acc = df_def['End Acc'].mean()
        std_nat_acc = df_def['End Acc'].std()
        max_poison_success = df_def['P1 Acc'].max() * 100


        if header is None:
            results_df.loc[defense, 'Avg Poison Success'] = f'{avg_poison_success:.2f}\u00B1{std_poison_success:.1f}'
            results_df.loc[defense,'Avg Nat Acc'] = f'{avg_nat_acc:.2f}\u00B1{std_nat_acc:.2f}'
            results_df.loc[defense,  'Max Poison Success'] = f'{max_poison_success:.2f}'

        else:
            results_df.loc[defense, (header_idx, 'Avg Poison Success')] = f'{avg_poison_success:.2%}\u00B1{std_poison_success:.1f}'
            results_df.loc[defense, (header_idx, 'Avg Nat Acc')] = f'{avg_nat_acc:.2f}\u00B1{std_nat_acc:.1f}'
            results_df.loc[defense, (header_idx, 'Max Poison Success')] = f'{max_poison_success:.2f}'

    else:

        success_rate = (df_def["Success"] == True).sum() / len(df_def)
        avg_nat_acc = df_def["End Acc"].mean()
        std_nat_acc = df_def["End Acc"].std()

        if header is None:
            results_df.loc[defense, 'Poison Success'] = f'{success_rate:.2%}'
            results_df.loc[defense, 'Natural Accuracy'] = f'{avg_nat_acc:.2f}%\u00B1{std_nat_acc:.1f}'

        else:
            results_df.loc[defense, (header_idx, 'Poison Success')] = f'{success_rate:.2%}'
            results_df.loc[defense, (header_idx, 'Natural Accuracy')] = f'{avg_nat_acc:.2f}%\u00B1{std_nat_acc:.1f}'

def export_results(df, narcissus = False, header=None, filters = {}):

    # Iterate through the filters
    for key, value in filters.items():
        df = df[df[key] == value]

    # Initialize an empty DataFrame for results
    if narcissus:
        if header is None:
            columns = ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success']
        else:
            columns = pd.MultiIndex.from_product([np.sort(df[header].unique()), ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success']],
                                     names=[header, 'Metric'])

    else:
        if header is None:
            columns = ['Poison Success', 'Natural Accuracy']
        else:
            columns = pd.MultiIndex.from_product([df[header].unique(), ['Poison Success', 'Natural Accuracy']],
                                                names=[header, 'Metric'])

    # Define the defenses
    defenses = ['None', 'EBM','Diff','EBM_Diff']                                         
    
    # Populate the DataFrame
    results_df = pd.DataFrame(columns=columns)

    if header is None: header_idxs = [0]
    else: header_idxs = df[header].unique()

    for header_idx in header_idxs:

        for defense in defenses:
            if header is None:
                if defense == 'Diff':
                    for steps in np.sort(df['Steps'].unique()):
                        df_def = df[(df['Defense'] == defense) & (df['Steps'] == steps)]
                        process_filtered_df(results_df,df_def,defense+f'-{steps}',narcissus,header,header_idx)
                        print(f'Added {defense}-{steps} to results_df with num rows {len(df_def)}')
                elif defense == 'EBM_Diff':
                    for steps in np.sort(df['Steps'].unique()):
                        df_def = df[(df['Defense'] == defense) & (df['Steps'] == steps)]
                        process_filtered_df(results_df,df_def,defense+f'-{steps}',narcissus,header,header_idx)
                        print(f'Added {defense}-{steps} to results_df with num rows {len(df_def)}')
                else:
                    df_def = df[(df['Defense'] == defense)]
                    process_filtered_df(results_df,df_def,defense,narcissus,header,header_idx)
                    print(f'Added {defense} to results_df with num rows {len(df_def)}')

            else:
                df_def = df[(df['Defense'] == 'None') & (df[header] == header_idx)]
                process_filtered_df(results_df,df_def,defense,narcissus,header,header_idx)

    return results_df

In [4]:
df = pd.read_csv('/home/sunaybhat/results_EBM_Defense/From_Scratch/Narcissus/Results.csv')

df['Defense'] = df['Defense'].fillna('None')
df['Steps'] = df['subset_size'] = df['Args'].apply(lambda x: json.loads(x)['diff_steps'] if 'diff_steps' in x else np.nan)
df['Steps'] = df['Steps'].fillna(125)

df_results = export_results(df,narcissus=True)

df_results

Added None to results_df with num rows 10
Added EBM to results_df with num rows 10
Added Diff-10.0 to results_df with num rows 10
Added Diff-25.0 to results_df with num rows 10
Added Diff-50.0 to results_df with num rows 10
Added Diff-100.0 to results_df with num rows 10
Added Diff-125.0 to results_df with num rows 10
Added Diff-150.0 to results_df with num rows 10
Added Diff-175.0 to results_df with num rows 10
Added Diff-200.0 to results_df with num rows 10
Added Diff-250.0 to results_df with num rows 10
Added Diff-500.0 to results_df with num rows 10
Added EBM_Diff-10.0 to results_df with num rows 10
Added EBM_Diff-25.0 to results_df with num rows 10
Added EBM_Diff-50.0 to results_df with num rows 10
Added EBM_Diff-100.0 to results_df with num rows 10
Added EBM_Diff-125.0 to results_df with num rows 10
Added EBM_Diff-150.0 to results_df with num rows 10
Added EBM_Diff-175.0 to results_df with num rows 10
Added EBM_Diff-200.0 to results_df with num rows 10
Added EBM_Diff-250.0 to res

,Avg Poison Success,Avg Nat Acc,Max Poison Success
None,43.95±33.6,94.89±0.23,93.59
EBM,1.34±0.6,92.73±0.22,2.29
Diff-10.0,42.63±47.5,94.42±0.13,99.76
Diff-25.0,42.08±48.6,94.04±0.16,99.99
Diff-50.0,22.66±39.8,93.67±0.13,99.86
Diff-100.0,2.36±2.4,93.64±0.13,8.92
Diff-125.0,2.00±2.0,93.50±0.16,7.49
Diff-150.0,1.96±1.9,93.36±0.23,7.16
Diff-175.0,1.72±1.4,93.24±0.26,5.42
Diff-200.0,1.40±0.9,93.19±0.29,3.69


In [9]:
df['Logs'].apply(lambda x: json.loads(x)['train_loss'])[0]


[1.9179745479617887,
 1.4543709767139172,
 1.2407012272368916,
 1.0339778805022959,
 0.8892892341479621,
 0.7750748780834705,
 0.6866544254905428,
 0.627800113481024,
 0.5800987233591202,
 0.5474209582714169,
 0.5197739152194899,
 0.5031422742492403,
 0.4824650273146227,
 0.4665756484736567,
 0.44813149755873033,
 0.4442681756318378,
 0.43076425786975703,
 0.4244910193526227,
 0.41725032142056223,
 0.40715148069364643,
 0.4011640531937485,
 0.39429829000969374,
 0.38584966835615886,
 0.3810331524942842,
 0.3743264714394079,
 0.37221626152315407,
 0.36462125003032975,
 0.3605127510664713,
 0.3563685262066019,
 0.35731240348590304,
 0.34913173569437794,
 0.3489763981012432,
 0.34767301510209625,
 0.34432335826746946,
 0.34254109905199015,
 0.33746281491063745,
 0.3349005826522627,
 0.33470999283711317,
 0.32959385349622466,
 0.3272460234896911,
 0.3294612658603112,
 0.32370188302548647,
 0.32619049074247364,
 0.3233782731930313,
 0.3207666279790956,
 0.319612363415301,
 0.315635062373050